# Katzen oder Brötchen: Model Inference & Interactive Image Tester
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/postelzhuktetyana/katzen-oder-broetchen/blob/main/cat_or_bun_tester.ipynb)

**Coursework Project**: Computer Vision Image Classification using FastAI & Deep Transfer Learning  
**Companion to**: [`cat_or_bun_classifier.ipynb`](cat_or_bun_classifier.ipynb)  
**Trained Model Assets**:
- `models/cat_or_bun_model.pkl` (Baseline Softmax — `CategoryBlock`)
- `models/cat_or_bun_multilabel_model.pkl` (Independent Sigmoid — `MultiCategoryBlock`)

---

### Notebook Overview
This notebook provides a dedicated, lightweight **dual-model inference and diagnostic workspace** for the "Cat or Bun" project. It loads both the **Baseline Softmax** model (`models/cat_or_bun_model.pkl`) and the **Independent Sigmoid Multi-Label** model (`models/cat_or_bun_multilabel_model.pkl`) to compare their predictions side-by-side on any image.

### Key Capabilities
1. **Decoupled Dual-Model Inference**: Loads both serialized `.pkl` bundles via FastAI's `load_learner()` without requiring training datasets.
2. **Side-by-Side Softmax vs. Sigmoid Comparison**: Visualizes closed-world Softmax (`Cat + Bun = 100%`) alongside independent Sigmoid probabilities (`CAT`, `BUN`, `BOTH`, or `NEITHER` at a `50%` threshold).
3. **Out-of-Distribution & Ambiguity Diagnostics**: Highlights where Softmax overconfidently forces unrelated images (e.g. `bike.png`, `wurst.png`) into `Cat`/`Bun` while Sigmoid suppresses both, as well as ambiguous "cat loaf" edge cases (`katzebroetchen.jpg`).
4. **Multiple Ways to Choose Images**:
   - **Interactive Dropdown**: Select test images from `testing_images/` or any custom folder.
   - **Direct Path Entry**: Enter an absolute or relative path to any image file on your computer.
   - **Drag-and-Drop Browser Uploader**: Use `ipywidgets.FileUpload` to test new photos directly from your browser.
   - **Batch Folder Evaluation**: Process an entire folder of test images and render an annotated visual gallery grid.


## 1. Environment Setup & Library Imports

### 1. Practical Goal
Prepare the runtime environment for model inference by loading essential computer vision libraries, interactive widgets, and visualization tools.

### 2. Core ML Concept: Decoupling Training and Inference
In real-world machine learning engineering, model inference is isolated from the training pipeline. The training dependencies (large datasets, data augmentations, optimizer updates, loss functions) are not needed at runtime. Inference only requires the learned weights, network architecture, and evaluation preprocessing transforms.

### 3. Python & FastAI Implementation Details
* `import fastai; from fastai.vision.all import *`: Imports FastAI's vision inference components (`load_learner`, `PILImage`).
* `import ipywidgets as widgets`: Interactive UI widgets for file selection, buttons, and drag-and-drop file upload.
* `import matplotlib.pyplot as plt`: Visual plotting library for side-by-side image display and probability bar charts.


In [ ]:
import io
import warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Silence PyTorch's internal TorchScript deprecation warning triggered on FastAI import
warnings.filterwarnings("ignore", category=FutureWarning, message=r".*torch\.jit\.script.*")

import fastai
from fastai.vision.all import *

print(f"FastAI version: {fastai.__version__}")
print(f"PyTorch version: {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Active inference device: {device.upper()}")


## 2. Dual Model Bundle Inspection & Loading

### 1. Practical Goal
Locate and deserialize both trained model bundles — `models/cat_or_bun_model.pkl` (Baseline Softmax) and `models/cat_or_bun_multilabel_model.pkl` (Independent Sigmoid) — using FastAI's `load_learner()`, and verify each model's vocabulary.

### 2. Core ML Concept: Controlled A/B Model Comparison
`load_learner()` reconstructs each PyTorch architecture, restores its trained weights, and sets `model.eval()` for deterministic inference. Loading both bundles side-by-side lets us evaluate the exact same input image through both the closed-world Softmax head and the open-world independent Sigmoid head.

### 3. Python & FastAI Implementation Details
* `get_multi_labels(img_path)`: Defined in `__main__` before `load_learner()` so pickle deserialization of the multi-label `DataBlock` resolves cleanly.
* `learners`: Dictionary mapping human-readable model names to loaded FastAI `Learner` objects.


In [ ]:
def get_multi_labels(img_path):
    """Map 'cat'/'bun' directories to single-element lists and 'other' to [] (required for unpickling)."""
    lbl = parent_label(img_path)
    return [] if lbl == 'other' else [lbl]

MODEL_SPECS = {
    'Baseline (Softmax)': Path('models/cat_or_bun_model.pkl'),
    'Multi-Label (Sigmoid)': Path('models/cat_or_bun_multilabel_model.pkl'),
}

learners = {}
for model_name, model_path in MODEL_SPECS.items():
    if not model_path.exists():
        print(f"⚠️ Skipping '{model_name}': file not found at '{model_path}'.")
        continue

    file_size_mb = model_path.stat().st_size / (1024 * 1024)
    print(f"Found {model_name}: {model_path} ({file_size_mb:.2f} MB)")
    lrn = load_learner(model_path, cpu=(device == 'cpu'))
    learners[model_name] = lrn
    print(f"  -> Loaded! Architecture: {lrn.model.__class__.__name__} | Vocab: {list(lrn.dls.vocab)}")

if not learners:
    raise FileNotFoundError(
        "No model bundles found in 'models/'. "
        "Expected 'models/cat_or_bun_model.pkl' and/or 'models/cat_or_bun_multilabel_model.pkl'."
    )

print(f"\n✅ Ready for comparison with {len(learners)} model(s): {list(learners.keys())}")


## 3. Core Prediction Engine & Dual-Model Comparison Visualization

- **Practical Goal**: Run inference on any image across all loaded models (`Baseline (Softmax)` vs. `Multi-Label (Sigmoid)`) and display their predictions side-by-side.
- **Core ML Concept**: Comparing closed-world Softmax (`Cat + Bun = 100%`) directly against independent Sigmoid (`CAT`, `BUN`, `BOTH`, or `NEITHER` at a `50%` decision threshold) makes calibration differences on out-of-distribution and ambiguous images immediately visible.

In [ ]:
THRESH = 0.5

def _summarize_prediction(pred_raw, probs, vocab, thresh=THRESH):
    """Normalize single-label or multi-label FastAI prediction into verdict + probability dict."""
    prob_dict = {cat.capitalize(): float(probs[i]) for i, cat in enumerate(vocab)}
    is_multi = not isinstance(pred_raw, str)

    if is_multi:
        active = [cat.capitalize() for cat, p in zip(vocab, probs) if float(p) >= thresh]
        if len(active) == 2:
            verdict = "BOTH (Cat & Bun)"
        elif len(active) == 1:
            verdict = active[0].upper()
        else:
            verdict = "NEITHER (Other / Unknown)"
    else:
        verdict = str(pred_raw).upper()
    return verdict, prob_dict, is_multi

def _predict_all_models(img, thresh=THRESH):
    """Run inference on a PILImage across every loaded model in `learners`."""
    results = {}
    for model_name, lrn in learners.items():
        pred_raw, _, probs = lrn.predict(img)
        vocab = list(lrn.dls.vocab)
        verdict, prob_dict, is_multi = _summarize_prediction(pred_raw, probs, vocab, thresh=thresh)
        results[model_name] = {
            'verdict': verdict,
            'probs': prob_dict,
            'is_multi': is_multi,
        }
    return results

def predict_and_display(img_source, title_label=None, thresh=THRESH):
    """
    Run inference across all loaded models and display:
    Left: Input image preview.
    Right: Side-by-side horizontal probability bar charts for each model.
    """
    try:
        if isinstance(img_source, (bytes, bytearray)):
            img = PILImage.create(io.BytesIO(img_source))
        else:
            img = PILImage.create(img_source)
    except Exception as e:
        print(f"❌ Error loading image: {e}")
        return None

    results = _predict_all_models(img, thresh=thresh)
    display_title = title_label or (Path(img_source).name if isinstance(img_source, (str, Path)) else "Input Image")

    print("=" * 66)
    print(f" Image: {display_title} (thresh={thresh*100:.0f}%)")
    print("-" * 66)
    for model_name, res in results.items():
        verdict = res['verdict']
        prob_dict = res['probs']
        is_multi = res['is_multi']
        probs_str = " | ".join(f"{cat}: {p*100:5.1f}%" for cat, p in prob_dict.items())
        print(f" {model_name:<22} -> {verdict:<27} [{probs_str}]")
        if is_multi and verdict.startswith("BOTH"):
            print("   ⚠️ Both categories ≥ 50% (Ambiguous Cat Loaf Case!)")
        elif is_multi and verdict.startswith("NEITHER"):
            print("   ℹ️ Both categories < 50% (Neither Cat nor Bun)")
    print("=" * 66)

    n_models = len(results)
    fig, axes = plt.subplots(
        1, 1 + n_models,
        figsize=(4.5 + 4.8 * n_models, 4.2),
        gridspec_kw={'width_ratios': [1.05] + [1.25] * n_models}
    )
    if not isinstance(axes, (list, np.ndarray)):
        axes = [axes]

    ax_img = axes[0]
    ax_img.imshow(img)
    ax_img.set_title(f"Input: {display_title}", fontsize=11, fontweight='bold', pad=10)
    ax_img.axis('off')

    for ax_bar, (model_name, res) in zip(axes[1:], results.items()):
        prob_dict = res['probs']
        verdict = res['verdict']
        is_multi = res['is_multi']

        categories = list(prob_dict.keys())
        values = [prob_dict[c] * 100.0 for c in categories]
        colors = ['#ff8566' if c == 'Cat' else '#e6b800' for c in categories]

        bars = ax_bar.barh(categories, values, color=colors, edgecolor='#333333', height=0.45)
        if is_multi:
            ax_bar.axvline(thresh * 100.0, color='#444444', linestyle='--', linewidth=1.3, label=f'{thresh*100:.0f}% Threshold')
            ax_bar.legend(loc='lower right', fontsize=8.5)
        ax_bar.set_xlim(0, 108)
        ax_bar.set_xlabel('Independent Probability (%)' if is_multi else 'Softmax Confidence (%)', fontsize=9.5, labelpad=8)
        ax_bar.set_title(f"{model_name}\nVerdict: {verdict}", fontsize=10.5, fontweight='bold', pad=10)
        ax_bar.grid(axis='x', linestyle='--', alpha=0.5)

        for bar in bars:
            w = bar.get_width()
            ax_bar.text(w + 2.0, bar.get_y() + bar.get_height() / 2, f"{w:.1f}%",
                        va='center', ha='left', fontsize=9.5, fontweight='bold', color='#222222')

    plt.tight_layout()
    plt.show()
    return results

## 4. Quick Sanity Test on Sample Image

### 1. Practical Goal
Validate the inference pipeline against the tracked external test image `testing_images/katzebroetchen.jpg`.

### 2. Core ML Concept: Edge-Case Evaluation
Testing on external, uncurated images tests whether the model's learned feature representations generalize beyond web-scraped training data. `katzebroetchen.jpg` features an ambiguous hybrid scene specifically designed to evaluate classification under visual uncertainty.

### 3. Python & FastAI Implementation Details
* Checks if `testing_images/katzebroetchen.jpg` exists locally (or pulls from `data/cats_and_buns/` fallback).
* Calls `predict_and_display()`.


In [ ]:
# Test on the standard external test image if present
test_path = Path('testing_images/katzebroetchen.jpg')

if test_path.exists():
    print(f"Testing on external sample image: {test_path}")
    predict_and_display(test_path)
else:
    # Fallback to any image from the downloaded dataset
    dataset_images = get_image_files('data/cats_and_buns') if Path('data/cats_and_buns').exists() else L()
    if len(dataset_images) > 0:
        sample_img = dataset_images[0]
        print(f"Testing on sample dataset image: {sample_img}")
        predict_and_display(sample_img)
    else:
        print("No test images found. Use the interactive picker below to choose an image from your filesystem!")


## 5. Interactive Filesystem Image Picker

### 1. Practical Goal
Provide an interactive GUI widget allowing users to select any image from `testing_images/`, the `data/` directory, or by typing an arbitrary file path from their filesystem.

### 2. Core ML Concept: Human-in-the-Loop Model Auditing
Interactive visual inspectors allow researchers, students, and domain experts to test arbitrary edge cases interactively, diagnosing blind spots (such as lighting changes, background textures, or unusual camera angles) without writing repetitive Python scripts.

### 3. Python & FastAI Implementation Details
* `widgets.Dropdown`: Dropdown menu listing all discoverable image files.
* `widgets.Text`: Freeform text box to input any arbitrary file path on your computer.
* `widgets.Button`: Interactive execution trigger.
* `widgets.Output`: Dedicated container for rendering the prediction results dynamically.


In [ ]:
# Discover all image files across testing_images and data directories
discovered_files = []
for search_dir in [Path('testing_images'), Path('data/cats_and_buns')]:
    if search_dir.exists():
        discovered_files.extend(get_image_files(search_dir))

file_options = [(f"{f.parent.name}/{f.name}", str(f)) for f in discovered_files]
if not file_options:
    file_options = [("No discovered images", "")]

# Construct interactive widgets
dropdown = widgets.Dropdown(
    options=file_options,
    description='Select File:',
    layout=widgets.Layout(width='60%')
)

custom_path_input = widgets.Text(
    value='',
    placeholder='Or type any path on your machine (e.g. /home/user/Pictures/my_cat.jpg)',
    description='Custom Path:',
    layout=widgets.Layout(width='85%')
)

classify_btn = widgets.Button(
    description='🔍 Classify Image',
    button_style='primary',
    tooltip='Run model inference on the selected image',
    icon='search',
    layout=widgets.Layout(width='200px', height='38px')
)

output_area = widgets.Output()

def on_classify_clicked(b):
    with output_area:
        clear_output()
        # Priority: custom path if filled, otherwise dropdown selection
        target_path = custom_path_input.value.strip() or dropdown.value
        if not target_path:
            print("⚠️ Please select an image from the dropdown or enter a valid file path.")
            return
            
        img_p = Path(target_path)
        if not img_p.exists():
            print(f"❌ File not found at: {img_p.resolve()}")
            return
            
        predict_and_display(img_p, title_label=img_p.name)

classify_btn.on_click(on_classify_clicked)

# Display the control dashboard
print("📁 Choose an image from the dropdown or enter any path from your computer:")
display(widgets.VBox([
    dropdown,
    custom_path_input,
    classify_btn,
    output_area
]))


## 6. Drag-and-Drop In-Browser File Uploader

### 1. Practical Goal
Allow users to upload or drag-and-drop any image file directly from their desktop or file explorer into the Jupyter interface for instant classification.

### 2. Core ML Concept: Web-Like Prototyping
Interactive upload widgets bridge the gap between static notebooks and live web deployment (such as Gradio or Streamlit apps), allowing non-technical evaluators to interact directly with the model.

### 3. Python & FastAI Implementation Details
* `widgets.FileUpload(accept='image/*', multiple=False)`: Native browser file selector widget.
* Robust decoding supporting both `ipywidgets 7.x` (dictionary structure) and `ipywidgets 8.x` (tuple of file objects).
* Direct byte stream ingestion via `io.BytesIO()`.


In [ ]:
uploader = widgets.FileUpload(
    accept='image/*',      # Restrict to image file formats
    multiple=False,        # Single file upload
    description='📤 Upload Image',
    button_style='success',
    layout=widgets.Layout(width='220px', height='38px')
)

upload_output = widgets.Output()

def on_upload_change(change):
    with upload_output:
        clear_output()
        if not uploader.value:
            return
            
        file_bytes = None
        file_name = "Uploaded Image"
        
        # Handle both ipywidgets 7.x and 8.x formats
        val = uploader.value
        if isinstance(val, dict):  # ipywidgets 7.x
            first_key = next(iter(val))
            file_name = first_key
            file_bytes = val[first_key]['content']
        elif isinstance(val, (list, tuple)) and len(val) > 0:  # ipywidgets 8.x
            item = val[0]
            file_name = getattr(item, 'name', 'Uploaded Image')
            if isinstance(item, dict):
                file_bytes = item.get('content')
            elif hasattr(item, 'content'):
                file_bytes = bytes(item.content)
            elif hasattr(item, 'tobytes'):
                file_bytes = item.tobytes()
                
        if file_bytes:
            print(f"✓ Received uploaded file: '{file_name}' ({len(file_bytes)} bytes)")
            predict_and_display(file_bytes, title_label=file_name)
        else:
            print("⚠️ Could not read uploaded file content.")

uploader.observe(on_upload_change, names='value')

print("📤 Click below to upload or drag-and-drop an image from your computer:")
display(widgets.VBox([uploader, upload_output]))


## 7. Batch Evaluation & Annotated Gallery Grid

### 1. Practical Goal
Evaluate an entire folder of test images in a single batch pass, generating an annotated gallery grid displaying input images, predicted labels, and confidence tags.

### 2. Core ML Concept: Batch Inference & Error Surface Inspection
Auditing a diverse collection of images simultaneously allows quick visual inspection of error distributions and systematic biases (e.g. consistently mistaking round cinnamon buns for sleeping cats).

### 3. Python & FastAI Implementation Details
* `get_image_files(folder)`: Recursively retrieves all valid image file paths.
* Multi-panel Matplotlib grid layout with color-coded title badges (green for high-confidence predictions, amber for ambiguous cases).


In [ ]:
def test_directory_gallery(folder_path='testing_images', max_images=8, thresh=THRESH):
    """
    Evaluate all images in a target directory and render an annotated visual gallery grid.
    """
    p = Path(folder_path)
    if not p.exists():
        print(f"Directory '{p}' does not exist.")
        return

    imgs = get_image_files(p)[:max_images]
    if len(imgs) == 0:
        print(f"No image files found in '{p}'.")
        return

    n = len(imgs)
    cols = min(4, n)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.8, rows * 4.0))
    axes = np.array(axes).reshape(-1)

    print(f"📊 Batch Dual-Model Gallery: Evaluating {n} images from '{p}' across {len(learners)} model(s)...")

    for i, img_path in enumerate(imgs):
        ax = axes[i]
        try:
            pil_img = PILImage.create(img_path)
            results = _predict_all_models(pil_img, thresh=thresh)

            lines = [img_path.name[:22]]
            short_verdicts = []
            for model_name, res in results.items():
                short_name = "Softmax" if not res['is_multi'] else "Sigmoid"
                short_v = res['verdict'].split(" ")[0]
                short_verdicts.append(short_v)
                summary = " | ".join(f"{k[0]}:{v*100:.0f}%" for k, v in res['probs'].items())
                lines.append(f"{short_name}: {short_v} ({summary})")

            ax.imshow(pil_img)
            disagree = len(set(short_verdicts)) > 1
            has_ambig = any(v in ('BOTH', 'NEITHER') for v in short_verdicts)
            title_color = '#b35900' if (disagree or has_ambig) else '#006600'
            ax.set_title("\n".join(lines), fontsize=8.5, fontweight='bold', color=title_color)
            ax.axis('off')
        except Exception as e:
            ax.set_title(f"Error: {e}", fontsize=9, color='red')
            ax.axis('off')

    for j in range(n, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

# Run batch gallery evaluation on testing_images
test_directory_gallery('testing_images', max_images=8)